# P2.1 · Notebook de baselines — Cebada360
Reproducible: usa `ml/src/` (protocolo P2.2, contrato v0.3.0). Este notebook vive en `notebooks/` y localiza solo la carpeta `ml/`.

**Dataset:** define `CEBADA_DATA` (carpeta con `features/features_master.csv`) o coloca el dataset en `data/DATASET_RETO_AGRO_2026_LIMPIO/`. Si el ZIP de Data trae otro layout, conviértelo antes con `python ml/prepare_data.py --src <zip> --out data/DATASET_RETO_AGRO_2026_LIMPIO`.

**Parámetros por entorno:** `CEBADA_REPS` (repeticiones de CV; 10 = oficial, 1–3 = prueba rápida) y `CEBADA_FAST=1` (menos árboles; solo pruebas).

In [ ]:
import os, sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
# Localiza ml/ (donde está src/) subiendo desde el directorio actual: sirve desde notebooks/ o desde la raíz del repo.
HERE = Path.cwd().resolve()
ML = next((p / 'ml' for p in [HERE, *HERE.parents] if (p / 'ml' / 'src' / 'config.py').exists()), None) \
     or next((p for p in [HERE, *HERE.parents] if (p / 'src' / 'config.py').exists()), None)
assert ML is not None, 'No se encontró la carpeta ml/ (con src/config.py). Abre el notebook dentro del repositorio.'
sys.path.insert(0, str(ML))
import numpy as np, pandas as pd
from src import config as C
from src.data import load, families, columns_for, EXPERIMENTS
from src.models import zoo
from src.cv import repeated_cv, spatial_cv, paired_diff
from src import provenance as PV
REPS = int(os.environ.get('CEBADA_REPS', C.N_REPEATS)); FAST = os.environ.get('CEBADA_FAST', '0') == '1'
print('ml/ =', ML, '| datos =', C.DATA_ROOT, '| repeticiones =', REPS, '| fast =', FAST)
tr, ev = load(); fam = families(tr); y = tr[C.TARGET].values
st = PV.audit_state(); print('Auditoría temporal válida para este dataset:', st['valid'], '-', st['reason'])
print(tr.shape, ev.shape); {k: len(v) for k, v in fam.items()}

## 1. Auditoría rápida del target y de la partición

In [ ]:
print(tr[C.TARGET].describe())
print(tr.groupby('estado')[C.TARGET].agg(['count','mean','std']).round(2))
print(tr[C.TARGET].round(2).value_counts().head(6))  # valores redondeados (3.0, 3.5, 2.5...) => posible captura aproximada

In [ ]:
from sklearn.neighbors import NearestNeighbors
Xc = tr[['centroid_lat','centroid_lon']].values
nn = NearestNeighbors(n_neighbors=2).fit(Xc)
d_tt = nn.kneighbors(Xc)[0][:,1]*111
d_et = nn.kneighbors(ev[['centroid_lat','centroid_lon']].values, n_neighbors=1)[0][:,0]*111
print(f'km al vecino más cercano: train→train {np.median(d_tt):.2f} | eval→train {np.median(d_et):.2f}')  # si son similares, CV aleatoria ≈ examen

## 2. Baselines por familia de features (E0–E5)
CV 5-fold × `N_REPEATS`; imputación/escalado dentro del fold.

In [ ]:
models = zoo(fast=FAST)
rows = []
for e in EXPERIMENTS:
    cols = columns_for(e, fam)
    tipo = 'baseline' if e == 'E0_baseline' else ('oficial' if e in C.OFFICIAL_EXPERIMENTS else 'diagnostico')
    for name, mk in models.items():
        if (e == 'E0_baseline') != (name == 'dummy_median'): continue
        X = tr[cols].values if cols else np.zeros((len(tr),1))
        rep, oof, _ = repeated_cv(mk, X, y, n_repeats=REPS)
        sp,_ = spatial_cv(mk, X, y, tr['grupo_espacial'].values)
        rows.append(dict(exp=e, tipo=tipo, model=name, n_feat=len(cols), rmse=rep.rmse.mean(), rmse_sd=rep.rmse.std(ddof=1) if REPS > 1 else np.nan,
                         mae=rep.mae.mean(), r2=rep.r2.mean(), rmse_spatial=sp['rmse']))
lb = pd.DataFrame(rows).sort_values('rmse'); lb.round(3)

## 3. Tabla estándar de resultados (P2.2)
Columnas fijas: `exp, tipo, model, n_feat, rmse, rmse_sd, mae, r2, rmse_spatial`. `tipo`: **oficial** (E1–E5), **baseline** (E0) o **diagnostico** (X*, usan municipio/ubicación: no son candidatos al modelo oficial). Se guarda en `ml/results/leaderboard_notebook.csv`; el leaderboard oficial lo produce `ml/run_experiments.py`.

In [ ]:
out = ML / 'results'; out.mkdir(exist_ok=True)
lb.to_csv(out / 'leaderboard_notebook.csv', index=False); print('guardado en', out / 'leaderboard_notebook.csv')